# Create a virtual environment && load necessary packages.


In [9]:
!pip install virtualenv
!virtualenv bip
!source bip/bin/activate
!pip install z3-solver
!pip install onnx
!git clone https://github.com/ytsao/lattice-gym

created virtual environment CPython3.12.13.final.0-64-x86_64 in 502ms
  creator CPython3Posix(dest=/content/bip, clear=False, no_vcs_ignore=False, global=False)
  seeder FromAppData(download=False, pip=bundle, via=copy, app_data_dir=/root/.cache/virtualenv)
    added seed packages: pip==26.1.2
  activators BashActivator,CShellActivator,FishActivator,NushellActivator,PowerShellActivator,PythonActivator,XonshActivator
Cloning into 'lattice-gym'...
remote: Enumerating objects: 1615, done.
remote: Counting objects: 100% (60/60), done.
remote: Compressing objects: 100% (41/41), done.
remote: Total 1615 (delta 21), reused 47 (delta 16), pack-reused 1555 (from 1)
Receiving objects: 100% (1615/1615), 55.55 MiB | 21.65 MiB/s, done.
Resolving deltas: 100% (1206/1206), done.
Updating files: 100% (740/740), done.


In [2]:
from z3 import *
from z3.z3util import get_vars

# Tutorial 1: What is Z3?
Z3 is a satisfied modulo theory (SMT) solver developed by Microsoft.

SMT solver is more general than satisfied (SAT) solver.
It can solve general problems not only restricted to Boolean problems.
For example, xxx

For more details, you can check xxx

In [5]:
# Define a solver
solver = Solver()

# Declare variables
# Boolean variables
b1 = Bool("b1")
b2 = Bool("b2")
solver.add(Not(b1))
solver.add(And(b1, b2))
status = solver.check()
print("Status:", status)
if status == sat:
    print(solver.model())

solver.reset()

# Integer variables
i1 = Int("i1")
i2 = Int("i2")
i3 = Int("i3")
solver.add(And(i1 >= 0, i1 <= 10))
solver.add(And(i2 >= 0, i2 <= 10))
solver.add(And(i3 >= 0, i3 <= 10))
solver.add(i1 + i2 + i3 == 10)
status = solver.check()
print("Status:", status)
if status == sat:
    print(solver.model())
    # How to find all models?

solver.reset()

# Real variables
r1 = Real("r1")
r2 = Real("r2")
solver.add(And(r1 >= 0, r1 <= 10))
solver.add(And(r2 >= 0, r2 <= 10))
solver.add(r1 + r2 == 10)
status = solver.check()
print("Status:", status)
if status == sat:
    print(solver.model())

solver.reset()

# Floating-point variables
f1 = FP("f1", FPSort(11, 53))  # Double precision
solver.add(f1 > FPVal(5.0, FPSort(11, 53)))
status = solver.check()
print("Status:", status)
if status == sat:
    print(solver.model())




Status: unsat
Status: sat
[i2 = 0, i3 = 0, i1 = 10]
Status: sat
[r1 = 0, r2 = 10]
Status: sat
[f1 = 1.2500000000000002220446049250313080847263336181640625*(2**2)]


# Exercise 1: Solve sudoku in Z3

### Example:
```text
+-------+-------+-------+
| 5 3 . | . 7 . | . . . |
| 6 . . | 1 9 5 | . . . |
| . 9 8 | . . . | . 6 . |
+-------+-------+-------+
| 8 . . | . 6 . | . . 3 |
| 4 . . | 8 . 3 | . . 1 |
| 7 . . | . 2 . | . . 6 |
+-------+-------+-------+
| . 6 . | . . . | 2 8 . |
| . . . | 4 1 9 | . . 5 |
| . . . | . 8 . | . 7 9 |
+-------+-------+-------+
```

$\forall i, j \in \{1,...,9\} \quad 1 \leq x_{i,j} \land x_{i,j} \leq 9 $

$x_{1,1} = 5 \land x_{1,2} = 3 \land x_{1,5} = 7$

$x_{2,1} = 6 \land x_{2,4} = 1 \land x_{2,5} = 9 \land x_{2,6} = 5$

$x_{3,2} = 9 \land x_{3,3} = 8 \land x_{3,8} = 6$

$x_{4,1} = 8 \land x_{4,5} = 6 \land x_{4,9} = 3$

$x_{5,1} = 4 \land x_{5,4} = 8 \land x_{5,6} = 3 \land x_{5,9} = 1$

$x_{6,1} = 7 \land x_{6,5} = 2 \land x_{6,9} = 6$

$x_{7,2} = 6 \land x_{7,7} = 2 \land x_{7,8} = 8$

$x_{8,4} = 4 \land x_{8,5} = 1 \land x_{8,6} = 9 \land x_{8,9} = 5$

$x_{9,5} = 8 \land x_{9,8} = 7 \land x_{9,9} = 9$

$\forall i \in \{1,...,9\} \forall j \in \{1,...,9\} \quad x_{i,j} \neq x_{i,j}$

$\forall i \in \{1, 4, 7\} \forall j \in \{1, 4, 7\} \quad x_{i,j} \neq x_{i,j}$

In [ ]:
def exercise1():
    # TODO:
    return

# Tutorial 2: How to verify neural networks by using z3?

![alt text](./images/nnv.png "nnv example")


**preconditions:**

$1 \leq x_1 \land x_1 \leq 2$

$2 \leq x_2 \land x_2 \leq 3$

**hidden layer:**

$h_1 = 0.8 * x_1 - 0.7 * x_2$

$h_2 = 0.6 * x_1 + 0.5 * x_2$

**Relu layer:**

$a_1 = ReLU(h_1, 0.0) \iff max(h_1, 0.0)$

$a_1 = ReLU(h_2, 0.0) \iff max(h_2, 0.0)$

But, there is no `max` function in Z3 Python API can be used.

How to deal with it?

**output layer & postcondition:**

$o_1 = -1.0 * a_1 + 0.4 * a_2$

$o_1 < 0.5$

In [13]:
def verify():
  verifier = Solver()
  x1 = Real("x1")
  x2 = Real("x2")
  verifier.add(And(1.0 <= x1, x1 <= 2.0))
  verifier.add(And(2.0 <= x2, x2 <= 3.0))
  h1 = Real("h1")
  h2 = Real("h2")
  verifier.add(h1 == 0.8*x1 - 0.7*x2)
  verifier.add(h2 == 0.6*x1 + 0.5*x2)
  a1 = Real("a1")
  a2 = Real("a2")
  verifier.add(a1 == If(h1 >= 0.0, h1, 0.0))
  verifier.add(a2 == If(h2 >= 0.0, h2, 0.0))
  o1 = Real("o1")
  verifier.add(o1 == -1.0*a1 + 0.4*a2)
  verifier.add(o1 < 0.5)

  status = verifier.check()
  print("Status:", status)

  if status == sat:
    print(verifier.model())

  return

verify()

Status: unsat


# Exercise 2:

In the previous practice, our code works!
However, it is not flexible to extend to other instances.

We provide different instances in the following table:
| ONNX Model File | VNNLIB Specification File |
| :--- | :--- |
| `instances/onnx/ToyNet.onnx` |     `instances/vnnlib/ToyNet.vnnlib` |
| `instances/onnx/test_nano.onnx` |  `instances/vnnlib/test_nano.vnnlib` |
| `instances/onnx/test_tiny.onnx` |  `instances/vnnlib/test_tiny.vnnlib` |
| `instances/onnx/test_small.onnx` | `instances/vnnlib/test_small.vnnlib` |
| `instances/onnx/sat_v6_c27.onnx` | `instances/vnnlib/sat_v6_c27.vnnlib` |


### What is onnx?



### What is vnnlib?


In [14]:
import onnx
from onnx import numpy_helper

def load_network(filename: str):
  # if the filename is existsed, load the onnx model and return it.
  if os.path.exists(filename):
    return onnx.load(filename)
  raise FileNotFoundError(f"File {filename} not found.")

def load_vnnlib(filename: str):
  # if the filename is existsed, load the vnnlib file and return it.
  if os.path.exists(filename):
    ctx = Context()
    constraints = parse_smt2_file(filename, ctx=ctx)
    return constraints
  raise FileNotFoundError(f"File {filename} not found.")

class Layer:
  def __init__(self):
    self.type = None
    self.weights = None
    self.biases = None

In [18]:
def exercise2(network, properties):
  # Verify the network with the given network and properties, return the verification results.
  # iterate node in the network graph
  def build(network):
    mp_init = {}
    for init in network.graph.initializer:
      mp_init[init.name] = init

    layers = []
    for node in network.graph.node:
      layer = Layer()
      layer.type = node.op_type
      for i in node.input:
        if i in mp_init.keys():
          tensor = numpy_helper.to_array(mp_init[i])
          if tensor.ndim == 2:
            layer.weights = tensor
          elif tensor.ndim == 1:
            layer.biases = tensor
      layers.append(layer)
    return layers

  layers = build(network)
  for layer in layers:
    print(f"Layer type: {layer.type}")
    print(f"Weights: {layer.weights}")
    print(f"Biases: {layer.biases}\n")

    if layer.type == "MatMul":
      # TODO:
      print(f"""
MutMul layer processing not implemented yet.
Weights shape: {layer.weights.shape}
Weights: {layer.weights}
""")
    elif layer.type == "Gemm":
      # TODO:
      print(f"""
Gemm layer processing not implemented yet.
Weights shape: {layer.weights.shape}
Weights: {layer.weights}
Biases shape: {layer.biases.shape}
Biases: {layer.biases}
""")
    elif layer.type == "Relu":
      # TODO:
      print("""Relu layer processing not implemented yet.
There is no weights or biases for Relu layer.""")
    elif layer.type == "Add":
      # TODO:
      print("""Add layer processing not implemented yet.
Biases shape: {layer.biases.shape}
Biases: {layer.biases}
""")
    else:
      raise NotImplementedError(f"Layer type {layer.type} not implemented yet.")

  return

network = load_network("/content/lattice-gym/bip/instances/onnx/ToyNet.onnx")
properties = load_vnnlib("/content/lattice-gym/bip/instances/vnnlib/ToyNet.vnnlib")
exercise2(network, properties)

Layer type: Gemm
Weights: [[ 0.8 -0.7]
 [ 0.6  0.5]]
Biases: [0. 0.]


Gemm layer processing not implemented yet.
Weights shape: (2, 2)
Weights: [[ 0.8 -0.7]
 [ 0.6  0.5]]
Biases shape: (2,)
Biases: [0. 0.]

Layer type: Relu
Weights: None
Biases: None

Relu layer processing not implemented yet.
There is no weights or biases for Relu layer.
Layer type: Gemm
Weights: [[-1.   0.4]]
Biases: [0.]


Gemm layer processing not implemented yet.
Weights shape: (1, 2)
Weights: [[-1.   0.4]]
Biases shape: (1,)
Biases: [0.]



# Exercise 3: Apply forward bound propagation with interval abstraction to verify neural networks.

As we have seen in the morning, we can apply a general forward bound propagation to verify neural networks.

By doing so, we can verify neural networks without invoking any external solvers.

In [ ]:
class Interval:
  def __init__(self, lower, upper):
    self.lower = lower
    self.upper = upper

def interval_prop(network, properties):
  # Verify the network with the given network and properties, return the verification results.
  # iterate node in the network graph
  def build(network):
    mp_init = {}
    for init in network.graph.initializer:
      mp_init[init.name] = init

    layers = []
    for node in network.graph.node:
      layer = Layer()
      layer.type = node.op_type
      for i in node.input:
        if i in mp_init.keys():
          tensor = numpy_helper.to_array(mp_init[i])
          if tensor.ndim == 2:
            layer.weights = tensor
          elif tensor.ndim == 1:
            layer.biases = tensor
      layers.append(layer)
    return layers

  layers = build(network)
  for layer in layers:
    print(f"Layer type: {layer.type}")
    print(f"Weights: {layer.weights}")
    print(f"Biases: {layer.biases}")

    if layer.type == "MatMul":
      # TODO:
      print(f"""
MutMul layer processing not implemented yet.
Weights shape: {layer.weights.shape}
Weights: {layer.weights}
""")
    elif layer.type == "Gemm":
      # TODO:
      print(f"""
Gemm layer processing not implemented yet.
Weights shape: {layer.weights.shape}
Weights: {layer.weights}
Biases shape: {layer.biases.shape}
Biases: {layer.biases}
""")
    elif layer.type == "Relu":
      # TODO:
      print("""Relu layer processing not implemented yet.
There is no weights or biases for Relu layer.""")
    elif layer.type == "Add":
      # TODO:
      print("""Add layer processing not implemented yet.
Biases shape: {layer.biases.shape}
Biases: {layer.biases}
""")
    else:
      raise NotImplementedError(f"Layer type {layer.type} not implemented yet.")

  return

network = load_network("instances/onnx/ToyNet.onnx")
properties = load_vnnlib("instances/vnnlib/ToyNet.vnnlib")
interval_prop(network, properties)

Layer type: Gemm
Weights: [[ 0.8 -0.7]
 [ 0.6  0.5]]
Biases: [0. 0.]

Gemm layer processing not implemented yet.
Weights shape: (2, 2)
Weights: [[ 0.8 -0.7]
 [ 0.6  0.5]]
Biases shape: (2,)
Biases: [0. 0.]

Layer type: Relu
Weights: None
Biases: None
Relu layer processing not implemented yet.
There is no weights or biases for Relu layer.
Layer type: Gemm
Weights: [[-1.   0.4]]
Biases: [0.]

Gemm layer processing not implemented yet.
Weights shape: (1, 2)
Weights: [[-1.   0.4]]
Biases shape: (1,)
Biases: [0.]

